# Act-PRMs with Tinker — sampling, rewarding, and training via the API

*Companion notebook for* **[On Learning to Think with Action Process Reward Models](https://openreview.net/forum?id=2zsteCP2wy)** *(ICML 2026 Workshop RLxF).*

This is the **Tinker** counterpart to `act_prm_transformers.ipynb` — the same algorithm, but run the
way the paper's experiments actually ran: no local GPU, with
[Tinker](https://thinkingmachines.ai/blog/announcing-tinker/) hosting the LoRA training client and
serving three primitives that map one-to-one onto Act-PRM's needs:

| Act-PRM step | Tinker primitive |
|---|---|
| **E-step**: sample $G$ thoughts $z^{(g)} \sim p_{\theta^{(n)}}(z \mid s)$ | `sampling_client.sample_async` |
| **Reward**: $\tilde r(z) = p_{\theta^{(n)}}(x \mid s, z)$ over action tokens | `sampling_client.compute_logprobs_async` |
| **M-step**: $\sum_g \bar r(z^{(g)}) \nabla_\theta \log p_\theta(x, z^{(g)} \mid s)$ | `training_client.forward_backward_async(loss_fn="importance_sampling")` |

**Prerequisites**: a `TINKER_API_KEY` (e.g. in a local `.env`). Model + LoRA config follow the paper:
Qwen3-4B-Instruct-2507, rank 8.


## 0. Setup

In [ ]:
%pip install -q tinker python-dotenv "transformers>=4.51" numpy torch

In [ ]:
import os, textwrap
import numpy as np
import torch
from dotenv import load_dotenv

load_dotenv()  # expects TINKER_API_KEY=...
assert os.environ.get("TINKER_API_KEY"), "Set TINKER_API_KEY in your environment or .env"

import tinker
from tinker import types

MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"
LORA_RANK = 8

service_client = tinker.ServiceClient()
training_client = await service_client.create_lora_training_client_async(
    MODEL_NAME, rank=LORA_RANK
)
print("training client ready:", MODEL_NAME, f"(LoRA r={LORA_RANK})")

In [ ]:
# The HF tokenizer supplies the chat template; Tinker consumes raw token ids.
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def apply_template(messages, continue_final_message=False, add_generation_prompt=False):
    return tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=add_generation_prompt,
        continue_final_message=continue_final_message,
        enable_thinking=False,
    )

## 1. The data: an action-only trajectory

Same abridged Snorkel Finance step as the Transformers notebook. In the full pipeline these come from
HF datasets of successful rollouts (e.g. `mzio/aprm-snorkelai_agent_finance_reasoning`) with the
original thoughts stripped (`actions_only=True` in `environments/act_prm/utils.py`), long
observations optionally elided (`hide_observations`).

In [ ]:
SYSTEM_PROMPT = "You are a helpful assistant."
ACT_PRM_SYSTEM_PROMPT = (
    "You are a helpful assistant that infers reasoning thoughts behind your own observed actions."
)
THOUGHT_BOS, THOUGHT_EOS = "<thought>", "</thought>"

TRAJECTORY = [
    {"role": "user", "content": (
        "Here is the question: What is the company's lease financing strategy and how heavily "
        "does it rely on operating leases versus finance leases? "
        "The company to query in the database: meta"
    )},
    {"role": "assistant", "content":
        '<tool_call>\n{"name": "get_descriptions", "arguments": {"company_name": "meta"}}\n</tool_call>'},
    {"role": "user", "content": (
        "Result: 62 tables for 'meta', including "
        "'meta_LeaseBalanceSheetInformationTableTextBlock': lease assets and liabilities "
        "reported on the balance sheet."
    )},
    {"role": "assistant", "content":
        '<tool_call>\n{"name": "get_table_info", "arguments": {"company_name": "meta", '
        '"table_name": "meta_LeaseBalanceSheetInformationTableTextBlock"}}\n</tool_call>'},
]

STATE_MESSAGES = [{"role": "system", "content": SYSTEM_PROMPT}] + TRAJECTORY[:3]
TARGET_ACTION = TRAJECTORY[3]["content"]

## 2. E-step: sample thoughts with a sampling client

A sampling client snapshots the current weights. We prompt with the **reversal trick** — action shown
first, message truncated right after `<thought>\n` so the model continues into the thought (see the
blog's Appendix A / `generator/tinker_act_prompt_aprm.py:86-140`).

In [ ]:
sampling_client = await training_client.save_weights_and_get_sampling_client_async()

def build_action_prompted_tokens(state_messages, target_action):
    msgs = [{"role": "system", "content": ACT_PRM_SYSTEM_PROMPT}]
    msgs += [m for m in state_messages if m["role"] != "system"]
    msgs.append({"role": "assistant", "content": f"{target_action}\n\n{THOUGHT_BOS}\n"})
    return apply_template(msgs, continue_final_message=True)

G = 8              # thoughts per state (paper setting)
MAX_TOKENS = 256   # paper uses up to 1024

prompt = types.ModelInput.from_ints(build_action_prompted_tokens(STATE_MESSAGES, TARGET_ACTION))
result = await sampling_client.sample_async(
    prompt,
    num_samples=G,
    sampling_params=types.SamplingParams(
        max_tokens=MAX_TOKENS, temperature=1.0, stop=[THOUGHT_EOS],
    ),
)
thoughts = [
    tokenizer.decode(seq.tokens, skip_special_tokens=True).split(THOUGHT_EOS)[0].strip()
    for seq in result.sequences
]
for i, z in enumerate(thoughts):
    print(f"--- z^({i+1}) " + "-" * 60)
    print(textwrap.fill(z, 100), "\n")

## 3. Reward: action log-probs via `compute_logprobs_async`

Score in the **un-reversed** order — state, thought, action — under the task's *original* system
prompt. Tokenize the prefix (state + thought, `continue_final_message=True`) and the full sequence;
the suffix delta is the action tokens; the reward is their length-normalized probability
(`generator/tinker_act_prm.py:58-122`):

In [ ]:
async def action_likelihood(state_messages, thought, target_action):
    """~r(z) = exp(mean logprob of action tokens | state, thought). Also returns
    the full token list + per-token logprobs (reused as old_logprobs for the M-step)."""
    prefix_msgs = state_messages + [{"role": "assistant", "content": thought}]
    prefix_tokens = apply_template(prefix_msgs, continue_final_message=True)
    full_msgs = state_messages + [
        {"role": "assistant", "content": f"{thought}\n\n{target_action}"}
    ]
    full_tokens = apply_template(full_msgs)
    n_action = len(full_tokens) - len(prefix_tokens)

    full_logprobs = await sampling_client.compute_logprobs_async(
        types.ModelInput.from_ints(full_tokens)
    )
    action_lp = np.array(full_logprobs[-n_action:], dtype=np.float64)
    reward = float(np.exp(action_lp.mean()))          # length-normalized
    return reward, full_tokens, full_logprobs

state_len = len(apply_template(STATE_MESSAGES, add_generation_prompt=True))

scored = [await action_likelihood(STATE_MESSAGES, z, TARGET_ACTION) for z in thoughts]
raw_rewards = [s[0] for s in scored]

# EM group normalization: r-bar_g = ~r_g / sum_g' ~r_g'   (reward_method="em")
Z = sum(raw_rewards)
rewards = [r / Z for r in raw_rewards]
best = int(np.argmax(rewards))

print(f"{'g':>3} {'~r':>10} {'r-bar':>8}")
for g, (r_raw, r_bar) in enumerate(zip(raw_rewards, rewards)):
    print(f"{g+1:>3} {r_raw:>10.4f} {r_bar:>8.3f}" + ("   <- z-hat" if g == best else ""))

## 4. M-step: `forward_backward` with importance-sampling loss

Each sampled thought becomes a `tinker.Datum`: full `(state + thought + action)` tokens shifted by
one, with the state prefix **zero-weighted** so gradient flows only through thought + action tokens.
`advantages` carries the EM reward $\bar r(z^{(g)})$; `logprobs` carries the sampler's logprobs for
the importance ratio (`trainer/rl.py:308-395`, `trainer/tinker/update.py:44-91`).

In [ ]:
def make_datum(full_tokens, full_logprobs, advantage, prefix_len):
    input_tokens  = full_tokens[:-1]
    target_tokens = full_tokens[1:]
    n_target      = len(target_tokens)
    n_prefix      = prefix_len - 1                      # shift by one for next-token targets
    n_gen         = n_target - n_prefix                 # thought + action tokens

    old_logprobs = [0.0] * n_prefix + list(full_logprobs[prefix_len:])
    advantages   = [0.0] * n_prefix + [float(advantage)] * n_gen

    return types.Datum(
        model_input=types.ModelInput.from_ints(input_tokens),
        loss_fn_inputs={
            "target_tokens": types.TensorData.from_torch(torch.tensor(target_tokens)),
            "logprobs":      types.TensorData.from_torch(torch.tensor(old_logprobs)),
            "advantages":    types.TensorData.from_torch(torch.tensor(advantages)),
        },
    )

data = [
    make_datum(full_tokens, full_logprobs, r_bar, state_len)
    for (r, full_tokens, full_logprobs), r_bar in zip(scored, rewards)
]

fwd_bwd_future = await training_client.forward_backward_async(
    data, loss_fn="importance_sampling"
)
optim_future = await training_client.optim_step_async(
    types.AdamParams(learning_rate=4e-5, beta1=0.9, beta2=0.95, eps=1e-8)
)
fwd_bwd_result = await fwd_bwd_future.result_async()
await optim_future.result_async()
print("one M-step done.")

## 5. The full EM loop

Iterate: fresh sampling client → walk the trajectory sampling + scoring thoughts (committing the best
$\hat z_t$ into context before moving on) → one policy-gradient update over all groups. This is
Algorithm 1; the codebase's `ActPrmTrainer.train` wraps exactly this with batching, replay buffers,
and evals.

In [ ]:
async def em_iteration(trajectory, n_iter, g=G):
    global sampling_client
    sampling_client = await training_client.save_weights_and_get_sampling_client_async()
    state = [{"role": "system", "content": SYSTEM_PROMPT}, trajectory[0]]
    data, relabelled = [], [trajectory[0]]

    action_indices = [i for i, m in enumerate(trajectory) if m["role"] == "assistant"]
    for t, idx in enumerate(action_indices):
        x_t = trajectory[idx]["content"]

        # E-step: sample G thoughts (action-prompted bootstrap)
        prompt = types.ModelInput.from_ints(build_action_prompted_tokens(state, x_t))
        res = await sampling_client.sample_async(
            prompt, num_samples=g,
            sampling_params=types.SamplingParams(max_tokens=MAX_TOKENS, temperature=1.0,
                                                 stop=[THOUGHT_EOS]),
        )
        zs = [tokenizer.decode(s.tokens, skip_special_tokens=True).split(THOUGHT_EOS)[0].strip()
              for s in res.sequences]

        # score + EM-normalize
        prefix_len = len(apply_template(state, add_generation_prompt=True))
        scored = [await action_likelihood(state, z, x_t) for z in zs]
        raw = [s[0] for s in scored]
        rbar = [r / sum(raw) for r in raw]
        best = int(np.argmax(rbar))
        print(f"[iter {n_iter} | t={t+1}] best ~r = {raw[best]:.4f}")

        data += [make_datum(ft, lp, a, prefix_len)
                 for (r, ft, lp), a in zip(scored, rbar)]

        # commit best thought + logged action; append next observation
        state.append({"role": "assistant", "content": f"{zs[best]}\n\n{x_t}"})
        relabelled.append(state[-1])
        if idx + 1 < len(trajectory):
            state.append(trajectory[idx + 1])
            relabelled.append(trajectory[idx + 1])

    # M-step
    fb = await training_client.forward_backward_async(data, loss_fn="importance_sampling")
    opt = await training_client.optim_step_async(types.AdamParams(learning_rate=4e-5))
    await fb.result_async(); await opt.result_async()
    return relabelled

for n in range(2):   # a couple of EM iterations on our toy trajectory
    relabelled_trace = await em_iteration(TRAJECTORY, n)

## 6. Distill back: relabelled traces → SFT (`cross_entropy`)

Stage 2 of the paper's recipe: relabel every action-only log with the trained model's best thoughts,
then SFT a **fresh** LoRA client on the synthetic full traces. In Tinker terms the datum swaps
`logprobs`/`advantages` for plain `weights` and the loss becomes `cross_entropy`
(`trainer/act_prm.py:98-129`):

In [ ]:
def make_sft_datum(full_tokens, prefix_len, weight=1.0):
    input_tokens  = full_tokens[:-1]
    target_tokens = full_tokens[1:]
    n_prefix = prefix_len - 1
    weights = [0.0] * n_prefix + [float(weight)] * (len(target_tokens) - n_prefix)
    return types.Datum(
        model_input=types.ModelInput.from_ints(input_tokens),
        loss_fn_inputs={
            "target_tokens": types.TensorData.from_torch(torch.tensor(target_tokens)),
            "weights":       types.TensorData.from_torch(torch.tensor(weights)),
        },
    )

# e.g., supervise the full relabelled conversation given its first user turn
sft_state = [{"role": "system", "content": SYSTEM_PROMPT}, relabelled_trace[0]]
sft_full = apply_template(sft_state + relabelled_trace[1:])
sft_prefix_len = len(apply_template(sft_state, add_generation_prompt=True))

sft_client = await service_client.create_lora_training_client_async(MODEL_NAME, rank=LORA_RANK)
fb = await sft_client.forward_backward_async(
    [make_sft_datum(sft_full, sft_prefix_len)], loss_fn="cross_entropy"
)
opt = await sft_client.optim_step_async(types.AdamParams(learning_rate=1e-4))
await fb.result_async(); await opt.result_async()
print("distill-back SFT step done.")

## Scaling this up

The companion codebase (`act-prm-tinker`) runs this at paper scale. The pieces map as:

| This notebook | Codebase |
|---|---|
| reversal prompt | `generator/tinker_act_prompt_aprm.py:86-140` |
| state-only variant (no action hint) | `generator/tinker_act_prm.py:205-233` |
| reward + EM normalization | `generator/tinker_act_prm.py:58-174, 253-266` |
| M-step datum + update | `trainer/rl.py:308-395`, `trainer/tinker/update.py:44-91` |
| distill-back SFT | `trainer/act_prm.py:98-129`, `trainer/act_prm_sft_rl.py` |
| action-only data loading, few-shot seeds | `environments/act_prm/env.py`, `environments/act_prm/utils.py` |

Example paper-scale invocation:

```bash
python main.py \
  --env_config act_prm/snorkel_finance_fs1 \
  --model_config hf_qwen3_4b_inst_2507 --lora_config r8_a16_qkvo \
  --generator_config aprm_qwen3_ap --trainer_config aprm_for_sft100 \
  --batch_size 16 --group_size 8 --learning_rate 4e-5 --actions_only
```

```bibtex
@inproceedings{anonymous2026on,
  title={On Learning to Think with Action Process Reward Models},
  author={Michael Zhang and Madison Ho},
  booktitle={ICML 2026 Workshop on RL from World Feedback},
  year={2026},
  url={https://openreview.net/forum?id=2zsteCP2wy}
}
```